# Lesson 03 - Agentic Design Patterns

ဒီအတန်းမှာ ကျွန်ုပ်တို့က ထိရောက်တဲ့ AI ကိုယ်စားလှယ်တွေ ဖန်တီးဖို့ အခြေခံ ဒီဇိုင်းနမူနာ သုံးခုကို ရှာဖွေကြမယ်။

1. **ရှင်းလင်းတဲ့ ကိုယ်စားလှယ်ညွှန်ပြချက်တွေ** — ကိုယ်စားလှယ်လှုပ်ရှားမှုကို ဦးဆောင်ပေးမယ့် တိကျပြီး အရေးကြီး ဒေသနများ ဖန်တီးခြင်း  
2. **Pydantic မော်ဒယ်တွေနဲ့ ဖွဲ့စည်းထားတဲ့ ထုတ်လွှင့်ချက်** — ကိုယ်စားလှယ်တွေက မျှော်မှန်းထားနိုင်ပြီး စစ်ဆေးထားတဲ့ ဒေတာကို ပြန်လည်ပေးစေခြင်း  
3. **တစ်ခုတည်းတာဝန်ရှိတဲ့ ကိုယ်စားလှယ်များ** — တစ်ခုချင်းစီကောင်းစွာလုပ်ဆောင်နိုင်တဲ့ အာရုံစူးစိုက်ထားတဲ့ ကိုယ်စားလှယ်များ ဒီဇိုင်းဆွဲခြင်း  

ကျွန်ုပ်တို့မှာ **ခရီးသွားရာဒေသ အကြံပြုသူ** စနစ်တစ်ခုအတွက် ဒီနမူနာတွေကို တစ်ဆင့်ချင်း အသုံးပြုပြီး တိုက်ဆိုင်ရာဒေသများညွှန်ပြခြင်း၊ ရရှိနိုင်မှုစစ်ဆေးခြင်း၊ နှင့် တွေ့ကြုံရမယ့် ကိစ္စများကို ကိုင်တွယ်နည်း တည်ဆောက်သွားမှာ ဖြစ်သည်။


## စတင်ပြင်ဆင်ခြင်း


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## ပုံစံ 1: ပေးပို့သူအတွက် ရှင်းလင်းသောညွှန်ကြားချက်များ

အထိရောက်ဆုံးသော ပုံစံမှာလည်း လွယ်ကူဆုံးဖြစ်သည်- သင့်ပေးပို့သူအတွက် ရှင်းလင်းပြီး အသေးစိတ်ညွှန်ကြားချက်များရေးသားခြင်းဖြစ်သည်။

ကောင်းမွန်သောညွှန်ကြားချက်များသည် အောက်ပါအချက်များကို ဖေါ်ပြသည်-
- **ဘယ်သူဖြစ်သနည်း** (အကျင့်သရုပ်နှင့်အသံ)
- **ဘာလုပ်သင့်သနည်း** (အဆင့်လိုက်တာဝန်ကျေပွန်မှုများ)
- **ဘယ်လိုနည်းဖြင့် အပြုအမူပြုသင့်သနည်း** (ကန့်သတ်ချက်များနှင့်ပုံစံ)

အောက်တွင်၊ ၎င်း ဖန်တီးသော တိုက်ရိုက်ညွှန်ကြားချက်များဖြင့် မည်သည့်ဖြေကြားချက်ကိုမဆို ဖွဲ့စည်းပေးသော ခရီးသွားကွန်ဆာ့ရှ်အေဂျင့်ကို ဖန်တီးထားသည်။


In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Pattern 2: Pydantic မော်ဒယ်များဖြင့် စနစ်တကျ ထုတ်လွှင့်မှု

အခမဲ့ပုံစံစာသားသည် စကားပြောချိန်တွင် အသုံးဝင်သော်လည်း အောက်ဆုံးစနစ်များတွင် စနစ်တကျသော ဒေတာများ လိုအပ်သည်။
**Pydantic မော်ဒယ်များ**ကို **ကိရိယာလုပ်ဆောင်ချက်**နှင့် တွဲဖက်ခြင်းဖြင့်-

- အေးဂျင့်၏ ထုတ်လွှင့်မှုအတွက် တိကျသော schema ကို သတ်မှတ်နိုင်သည်
- တုံ့ပြန်ချက်များကို အလိုအလျောက် စစ်ဆေးနိုင်သည်
- အေးဂျင့်ရလဒ်များကို အက်ပလီကေးရှင်း အခြေခံ နိုင်စွမ်းဖြင့် ပေါင်းစပ်နိုင်သည်

ထို့အပြင် အဲ့ဒီ အေးဂျင့်၏ အကြံပေးချက်များကို တကယ့်ဒေတာများဖြင့် အခြေခံစေဖို့ သွားရောက်ရာနေရာ အသေးစိတ်ကို ပြန်အပ်သော ကိရိယာကိုပါ မိတ်ဆက်ပေးပါသည်။


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = await provider.create_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## ပုံစံ ၃: တစ်ခုတည်းတာဝန်ရှိသော အေးဂျင့်များ

စည်းကမ်းကျကျ သီးခြားထားသော အေးဂျင့်များစွာအတွင်း အလုပ်ခွဲခြမ်းခြင်းဖြင့် ရိုးရှင်းသောတာဝန်တစ်ခုစီကို ပေးသော စိစစ်အလုပ်များမှ ရလဒ်ကောင်းများ ရရှိသည်-

-  တည်နေရာများနှင့် ရရှိနိုင်မှုကို သိရှိထားသော **တည်နေရာကျွမ်းကျင်သူ**
-  လေကြောင်း၊ ဟိုတယ်၊ ခရီးစဉ်များကို စီစဉ်ပေးသော **ကုန်ပစ္စည်း စီမံခန့်ခွဲသူ**

ဤသည်မှာ ဆော့ဖ်ဝဲ အင်ဂျင်နီယာကျောင်းစံသည် *စိတ်မဝင်စားမှုများ ခွဲခြားခြင်း* ကို နှိုင်းယှဉ်၍ - အေးဂျင့်တိုင်းကို စမ်းသပ်ရန်၊ စောင့်ရှောက်ရန်နှင့် တိုးတက်အောင်လုပ်ရန် ပိုမိုလွယ်ကူစေသည်။


In [ ]:
destination_agent = await provider.create_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = await provider.create_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## Summary

ဒီသင်ခန်းစာမှာ ကျွန်တော်တို့ သုံးခုသော agentic ဒီဇိုင်းပုံစံများကို ခရီးသွား အကြံပြုသူ အခြေအနေတစ်ခုတွင် လျှောက်ထားခဲ့ပါတယ်-

| Pattern | Key Idea | Benefit |
|---|---|---|
| **Clear Instructions** | ပုဂ္ဂိုလ်ရေး၊တာဝန်များနှင့် ကန့်သတ်ချက်များကို ရှင်းလင်းစွာ သတ်မှတ်ခြင်း | လိုက်လျောညီထွေ၊ အမှတ်တံဆိပ်နှင့် ကိုက်ညီသော agent အပြုအမှု |
| **Structured Output** | တုံ့ပြန်ပုံစံအဖြစ် Pydantic မော်ဒယ်များကို အသုံးပြုခြင်း | စစ်ဆေးပြီး၊ စက်ဖတ်နိုင်သော အဖြေများ |
| **Single Responsibility** | အာရုံစူးစိုက်ထားသော တစ်ခုတည်းသော တာဝန်ကို ပေးခြင်း | စမ်းသပ်ရန်၊ ထိန်းသိမ်းရန် နှင့် ပေါင်းစည်းရန် လွယ်ကူခြင်း |

ဒီပုံစံတွေက သဘာဝကျကျ ပေါင်းစည်းနိုင်ပြီး — သေချာရှင်းလင်းသော သတင်းအချက်အလက်တွေနဲ့ ပေါင်းစပ်ထားတဲ့ တစ်ခုတည်းတာဝန် agent အတွင်းတွင် သင်ကြားပေးပြီး သို့ရောက်မြောက်တဲ့ ဖက်တိုးပြီး အသုံးပြုနိုင်သော စနစ်များ ဖန်တီးနိုင်ပါတယ်။


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免责声明**  
ဤစာတမ်းကို AI ဘာသာပြန်ဝန်ဆောင်မှု [Co-op Translator](https://github.com/Azure/co-op-translator) အသုံးပြု၍ ဘာသာပြန်ထားသည်။ ကျွန်ုပ်တို့သည် တိကျမှန်ကန်မှုအတွက် ကြိုးစားပေမယ့်၊ အလိုအလျှောက်ဘာသာပြန်မှုတွင် အမှားများ သို့မဟုတ် မှားယွင်းမှုများ ပါဝင်နိုင်ကြောင်း ကျေးဇူးပြု၍ သတိပြုပါ။ မူလစာတမ်း၏ မိခင်ဘာသာစကားဖြင့် ရေးထားသည့် မူရင်းစာတမ်းကို တရားဝင်ရင်းမြစ်အနေဖြင့် ထည့်သွင်းစဉ်းစားရမည်ဖြစ်သည်။ အရေးကြီးသော သတင်းအချက်အလက်များအတွက် ကျွမ်းကျင်သော လူသားဘာသာပြန်မှုကို အကြံပြုပါသည်။ ဤဘာသာပြန်မှု အသုံးပြုခြင်းကြောင့် ဖြစ်နိုင်သည့် မတိကျမှုများ သို့မဟုတ် မှားယွင်းစွာ နားလည်မှုများအတွက် ကျွန်ုပ်တို့ တာဝန်မခံပါ။
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
